[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kamrullab/house-price-prediction-system/blob/main/notebooks/Dhaka_House_Rent_Prediction_BDT_Colab.ipynb)

# 🇧🇩 Dhaka House Rent Prediction — BDT (৳)

A complete, one-click machine-learning workflow using **3,083 real Dhaka rental listings**. The target is monthly asking rent in Bangladeshi Taka. Choose **Runtime > Run all**; no upload, Drive mount, API key, or path edit is needed.

> ⚠️ This predicts **monthly asking rent**, not a house sale price or certified valuation.

## 📦 Install and import packages

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'requests'], check=True)

from io import BytesIO
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
print('✅ Packages are ready')

## 🌐 Load the real Bangladesh dataset

Source: [Mendeley Data, DOI 10.17632/d3rxcxn74y.1](https://doi.org/10.17632/d3rxcxn74y.1), released under **CC0 1.0**.

In [ ]:
DATA_URL = 'https://data.mendeley.com/public-files/datasets/d3rxcxn74y/files/9bfbe702-69ef-4e1a-8ee6-922565feb251/file_downloaded'
response = requests.get(DATA_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=60)
response.raise_for_status()
data = pd.read_csv(BytesIO(response.content))
assert {'price_monthly_bdt', 'size_sqft', 'bedrooms', 'bathrooms', 'area_group'}.issubset(data.columns)
print(f'✅ Loaded {len(data):,} real Dhaka rental listings')
display(data.head())
display(data[['price_monthly_bdt', 'size_sqft', 'bedrooms', 'bathrooms']].describe().T)

## 📊 Explore rent, area, and floor size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.histplot(data=data, x='price_monthly_bdt', bins=50, ax=axes[0], color='#006A4E')
axes[0].set(title='Monthly Asking Rent', xlabel='Rent (৳/month)', xlim=(0, data.price_monthly_bdt.quantile(.98)))
sample = data.sample(min(2000, len(data)), random_state=42)
sns.scatterplot(data=sample, x='size_sqft', y='price_monthly_bdt', alpha=.4, ax=axes[1], color='#F42A41')
axes[1].set(title='Floor Size vs Rent', xlabel='Size (sq ft)', ylabel='Rent (৳/month)', ylim=(0, data.price_monthly_bdt.quantile(.98)))
top_areas = data.groupby('area_group')['price_monthly_bdt'].median().nlargest(10).sort_values()
top_areas.plot.barh(ax=axes[2], color='#006A4E')
axes[2].set(title='Top Areas by Median Rent', xlabel='Median rent (৳/month)', ylabel='Area')
plt.tight_layout(); plt.show()

## 🧠 Preprocess, train, and compare models

In [ ]:
numeric_features = ['size_sqft', 'bedrooms', 'bathrooms', 'views', 'num_images']
categorical_features = ['area_group', 'property_type_group', 'furnishing', 'condition', 'listing_by']
binary_features = ['is_promoted', 'has_lift', 'has_parking', 'has_generator', 'has_gas', 'has_balcony', 'has_ac', 'has_security', 'has_servant_room', 'has_gym_pool', 'has_view']
features = numeric_features + categorical_features + binary_features
X, y = data[features].copy(), data['price_monthly_bdt'].copy()
X[binary_features] = X[binary_features].astype('float64')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
binary_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))])
preprocessor = ColumnTransformer([('num', numeric_pipe, numeric_features), ('cat', categorical_pipe, categorical_features), ('bin', binary_pipe, binary_features)])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1),
}
results, fitted = [], {}
for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    fitted[name] = (pipeline, prediction)
    results.append({'Model': name, 'MAE (৳)': mean_absolute_error(y_test, prediction), 'RMSE (৳)': mean_squared_error(y_test, prediction) ** .5, 'R²': r2_score(y_test, prediction)})
results_df = pd.DataFrame(results).sort_values('RMSE (৳)').reset_index(drop=True)
display(results_df.style.format({'MAE (৳)': '৳{:,.0f}', 'RMSE (৳)': '৳{:,.0f}', 'R²': '{:.4f}'}))

## 🎯 Evaluate the best model

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model, best_prediction = fitted[best_name]
residuals = y_test.to_numpy() - best_prediction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=y_test, y=best_prediction, alpha=.45, ax=axes[0], color='#006A4E')
limits = [min(y_test.min(), best_prediction.min()), max(y_test.max(), best_prediction.max())]
axes[0].plot(limits, limits, '--', color='#F42A41')
axes[0].set(title=f'Actual vs Predicted — {best_name}', xlabel='Actual rent (৳/month)', ylabel='Predicted rent (৳/month)', xlim=(0, y_test.quantile(.98)), ylim=(0, y_test.quantile(.98)))
sns.scatterplot(x=best_prediction, y=residuals, alpha=.45, ax=axes[1], color='#F42A41')
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(title='Residual Plot', xlabel='Predicted rent (৳/month)', ylabel='Residual (৳)')
plt.tight_layout(); plt.show()

## 💰 Predict one Dhaka property's monthly rent

In [ ]:
example = pd.DataFrame([{
    'size_sqft': 1500, 'bedrooms': 3, 'bathrooms': 2, 'views': 25, 'num_images': 6,
    'area_group': 'Dhanmondi', 'property_type_group': 'Apartment', 'furnishing': 'Unfurnished',
    'condition': 'Fairly Used', 'listing_by': 'Owner', 'is_promoted': False,
    'has_lift': True, 'has_parking': True, 'has_generator': True, 'has_gas': True,
    'has_balcony': True, 'has_ac': False, 'has_security': True, 'has_servant_room': False,
    'has_gym_pool': False, 'has_view': False
}])[features]
example[binary_features] = example[binary_features].astype('float64')
prediction_model_name = 'Random Forest'
prediction_model = fitted[prediction_model_name][0]
estimated_rent = prediction_model.predict(example)[0]
print(f'🏠 Prediction model: {prediction_model_name}')
print(f'🇧🇩 Estimated monthly asking rent: ৳{estimated_rent:,.0f}/month')
print('ℹ️ Educational estimate only; actual negotiated rent can differ.')

## ✅ Finished

The notebook downloaded real CC0 Dhaka data, explored it, handled missing values, encoded categories, compared two models, evaluated the best one, and produced a BDT/month prediction. Dataset limitations include asking rather than contracted rents, one-day cross-sectional collection, neighbourhood imbalance, and missing listing attributes.